In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest,chi2,RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt



def rfeFeature (indep_X, dep_Y, n):
    
    from sklearn.linear_model import LinearRegression
    from sklearn.svm import SVR
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.ensemble import RandomForestRegressor
    
    lin = LinearRegression()
    SVRl = SVR(kernel = 'linear')
    dec = DecisionTreeRegressor(random_state = 0)
    rf = RandomForestRegressor(n_estimators = 10, random_state = 0)
    
    rfelist=[]
    rfemodellist=[lin,SVRl,dec,rf] 
    
    for i in   rfemodellist:
        print(i)
        log_rfe = RFE(i, n_features_to_select=n)
        log_fit = log_rfe.fit(indep_X, dep_Y)
        log_rfe_feature=log_fit.transform(indep_X)
        rfelist.append(log_rfe_feature)
    return rfelist

def split_scalar(indep_X,dep_Y):
  
    X_train,X_test,y_train,y_test=train_test_split(indep_X,dep_Y , test_size=0.25 , random_state=0)
    sc=StandardScaler()
    X_train=sc.fit_transform(X_train)
    X_test=sc.transform(X_test)
    return X_train,X_test,y_train,y_test

def r2_prediction(regressor,X_test,y_test):
    y_pred=regressor.predict(X_test)
    from sklearn.metrics import r2_score
    r2=r2_score(y_test,y_pred)
    return r2

def Linear (X_train,X_test,y_train,y_test):
    
    from sklearn.linear_model import LinearRegression
    regressor=LinearRegression()
    regressor.fit(X_train,y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return r2

def svm_linear(X_train,X_test,y_train,y_test):
                
    from sklearn.svm import SVR
    regressor = SVR(kernel = 'linear')
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return r2

def svm_NL(X_train,X_test,y_train,y_test):
            
    from sklearn.svm import SVR
    regressor = SVR(kernel = 'rbf')
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2 

def Decision(X_train,X_test,y_train,y_test):
        
    from sklearn.tree import DecisionTreeRegressor
    regressor = DecisionTreeRegressor(random_state = 0)
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2

def random(X_train,X_test,y_train,y_test):   
    
    from sklearn.ensemble import RandomForestRegressor
    regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2 


def rfe_regression(acclin,accsvml,accsvmnl,accdes,accrf):

    dataframe=pd.DataFrame({'liner':acclin,
                            'svml':accsvml,
                            'svmnl':accsvmnl,
                            'Decision':accdes,
                            'Random':accrf} , 
                           index=['linear_REF','svr_REF','decision_REF','random_REF'])
    return dataframe



dataset1=pd.read_csv("prep.csv",index_col=None)
df2=dataset1
df2 = pd.get_dummies(df2, drop_first=True)

indep_X=df2.drop('classification_yes', axis=1)
dep_Y=df2['classification_yes']

acclin=[]
accsvml=[]
accsvmnl=[]
accdes=[]
accrf=[]

rfelist=rfeFeature(indep_X,dep_Y,3)

for i in rfelist:

    X_train,X_test,y_train,y_test=split_scalar(i,dep_Y)

    r2_lin=Linear(X_train,X_test,y_train,y_test)
    acclin.append(r2_lin)
    
    r2_sl=svm_linear(X_train,X_test,y_train,y_test)    
    accsvml.append(r2_sl)
    
    r2_NL=svm_NL(X_train,X_test,y_train,y_test)
    accsvmnl.append(r2_NL)
    
    r2_d=Decision(X_train,X_test,y_train,y_test)
    accdes.append(r2_d)
    
    r2_r=random(X_train,X_test,y_train,y_test)
    accrf.append(r2_r)

result=rfe_regression(acclin,accsvml,accsvmnl,accdes,accrf)

    

LinearRegression()
SVR(kernel='linear')
DecisionTreeRegressor(random_state=0)
RandomForestRegressor(n_estimators=10, random_state=0)


In [3]:
rfelist

[array([[1., 0., 0.],
        [1., 0., 0.],
        [0., 0., 0.],
        ...,
        [1., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]], shape=(399, 3)),
 array([[1., 0., 0.],
        [1., 0., 0.],
        [0., 0., 0.],
        ...,
        [1., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]], shape=(399, 3)),
 array([[12.51815562,  1.        ,  0.        ],
        [10.7       ,  1.        ,  0.        ],
        [12.        ,  0.        ,  0.        ],
        ...,
        [ 9.1       ,  1.        ,  0.        ],
        [ 8.5       ,  0.        ,  0.        ],
        [16.3       ,  0.        ,  0.        ]], shape=(399, 3)),
 array([[ 3.        , 12.51815562,  0.        ],
        [ 2.        , 10.7       ,  0.        ],
        [ 1.        , 12.        ,  0.        ],
        ...,
        [ 3.        ,  9.1       ,  0.        ],
        [ 0.        ,  8.5       ,  0.        ],
        [ 0.        , 16.3       ,  0.        ]], shape=(399, 3))]

In [7]:
result

,liner,svml,svmnl,Decision,Random
linear_REF,0.441961,0.262153,0.262162,0.441961,0.441816
svr_REF,0.441961,0.262153,0.262162,0.441961,0.441816
decision_REF,0.664893,0.609652,0.883134,0.965961,0.916304
random_REF,0.676174,0.670539,0.900941,0.933504,0.887256
